# Labwork 3 — Stochastic optimization

**Week 2 · Day 3 · ≈ 170 min at the keyboard**

Read Lecture 3 first. This labwork fills in part of the `optlab` package you cloned;
you edit the real source files, and the notebook checks your work as you go.

**What you build today:** mini-batch training: the way essentially every modern machine-learning model is fitted

**Files you will open:**

- `problems/glm.py` (the `BatchObjective` half)
- `optimizers/stochastic.py`

> **The rule.** `src/optlab/interfaces.py`, `results.py`, `errors.py` and `types.py` are
> **provided** — never edit them. Everything else under `src/optlab/` is yours: replace
> each `raise NotImplementedError` with working code, keeping the signature and honouring
> the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone and gives you a
`check()` helper that runs a specific test file and reports what happened.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
print("optlab:", OPTLAB)


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

---

## Exercise 1 — The mini-batch gradient  *(≈ 45 min)*

Implement `GLMLoss.n_samples` and `GLMLoss.batch_gradient(w, idx)`.

Then do the refactor that matters: rewrite `gradient` so it **calls** `batch_gradient`
over all indices. The formula `Xᵀφ'/n` should appear exactly once in the file. If it
appears twice, the two copies will disagree the first time you change one.

Then measure the two properties from the lecture: draw many random batches at a fixed `w`
and confirm the mean error is ~0 (unbiased) and the spread falls like `1/√b`.

**Open:** `src/optlab/problems/glm.py`

In [ ]:
edit("problems/glm.py")
check("-m", "day3")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

`batch_gradient` is `X[idx].T @ d1(X[idx] @ w, y[idx]) / len(idx)`. Then `gradient(w)` is `batch_gradient(w, np.arange(self.n_samples))`.

</details>

---

## Exercise 2 — SGD  *(≈ 55 min)*

Implement `SGD.minimize`. Per epoch: reshuffle the indices, walk them in blocks of
`batch_size`, take a step on each block. The step at epoch `k` is `lr / (1 + lr_decay·k)`.

Use the **injected** `self.rng`. Never call `np.random.*` directly — that is what makes a
run reproducible from a seed, and it is dependency inversion applied to randomness.

Three things to verify:

1. **`batch_size = n_samples` must reproduce gradient descent exactly** with the same
   fixed step. If it does not, one of the two is wrong.
2. Two runs with the same seed give identical results; different seeds do not.
3. **The noise floor.** With a constant step, measure `f(w) − f*` at the plateau for
   `lr` and `lr/2`. Measure the *excess* over the optimum, not the raw loss — against the
   raw loss the effect is invisible. Then switch the decay on and watch the plateau
   disappear.

**Open:** `src/optlab/optimizers/stochastic.py`

In [ ]:
check("-m", "day3")

<details>
<summary><b>Plan B</b> — open only if you are stuck for more than ten minutes</summary>

For `f*`, run your day-2 gradient descent to a very tight tolerance first, and cache the value.

</details>

---

## Exercise 3 — Adam  *(≈ 45 min)*

Implement `Adam.minimize`: the two moment accumulators, the bias correction, and the
per-coordinate step.

Both moments start at zero, so both are biased toward zero for the first several steps —
divide by `1 - β₁ᵗ` and `1 - β₂ᵗ` to correct, where `t` counts **steps, not epochs**.

Two checks:

1. The first step is `≈ lr · sign(g)`, whatever the magnitude of `g`. Verify across six
   orders of magnitude.
2. Take your logistic problem, multiply **one** feature column by 1000, and compare. SGD
   is trapped: any step large enough to move the small-scale coordinates is unstable on
   the large-scale one. Adam barely notices.

That second result is the lecture's claim that Adam auto-standardizes the per-coordinate
scale — the optimizer-side answer to day 1's conditioning problem.

**Open:** `src/optlab/optimizers/stochastic.py`

In [ ]:
check("-m", "day3")

---

## Exercise 4 — Batch size against noise  *(≈ 25 min)*

Plot the loss against **epoch** (not iteration — equal data cost) for
`b ∈ {1, 32, 256, n}`, and SGD against Adam.

Record wall-clock as well as epochs. Small batches take more steps per epoch and make more
progress per epoch, but each step has fixed overhead, so the wall-clock winner is usually
neither extreme. Keep the numbers: they feed the day-6 benchmark.

**Open:** no file — work in the notebook

In [ ]:
print("Plot here: loss vs epoch for several batch sizes, then SGD vs Adam.")
print("Keep the numbers -- Labwork 6 reuses them.")

---

## Checkpoint

Everything from day 1 to day 3 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2 or day3")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

---

## Before the debrief

Be able to answer:

1. Why is the mini-batch gradient unbiased — and where does the argument use uniform
   sampling?
2. Why does a constant step not reach the exact optimum, when gradient descent with a
   constant step does?
3. What does Adam adapt *to*? Name the day-1 concept it compensates for.